In [3]:

import random
class Game:

    #initialise players and deck starting with default actions
    def __init__(self, players, deck):
        
        if len(players) not in [2, 3]:
            raise ValueError("The game requires 2 or 3 players")
            
        self.players = players
        self.deck = deck
        self.current_player = 0
        self.winner = None
        self.game_over = False
        self.turn_actions = {}

        self.start_turn()

        
    #what happens when a player's turn starts
    def start_turn(self):
        self.turn_actions = {
            "draw": False,
            "steal": False,
            "draw_2": False,
            "single_discards": 0
        }
    #funtion get_hand to access hands
    def _get_hand(self, player):
        return player.hand 

    #function for drawing a card from the deck
    def draw_card(self):
        if self.game_over:
            return False
        player = self.players[self.current_player]
        hand = self._get_hand(player)
        if self.turn_actions["draw"]:
            return False
        if len(hand)>=20:
            return False
        if len(self.deck) == 0:
            return False
        card = self.deck.pop()
        hand.append(card)
        self.turn_actions["draw"] = True
        return True
        
    def draw_2(self, keep_index):
        if self.game_over:
            return False     
    # Check whether this action has already been used
        if self.turn_actions["draw_2"]:
            return False

        player = self.players[self.current_player]
        hand = self._get_hand(player)
    
        # Check hand limit
        if len(hand) >= 20:
            return False
    
        # Check that at least two cards remain
        if len(self.deck) < 2:
            return False
    
        # Check which card the player wants to keep
        if keep_index not in [0, 1]:
            return False
    
        # Draw two cards
        drawn_cards = [self.deck.pop(), self.deck.pop()]
    
        # Keep the selected card
        kept_card = drawn_cards[keep_index]
        discarded_card = drawn_cards[1 - keep_index]
    
        hand.append(kept_card)
    
        # Return the unwanted card to the deck
        self.deck.append(discarded_card)
    
        # Reshuffle the deck
        random.shuffle(self.deck)
    
        # Record that the action has been used
        self.turn_actions["draw_2"] = True
    
        return True

    #function for checking whether a steal is possible
    def can_steal_card(self, target_index, card_index):  
        if self.game_over:
            return False
        if target_index < 0 or target_index >= len(self.players):
            return False
        current_player = self.players[self.current_player]
        target_player = self.players[target_index]
        
        current_hand = self._get_hand(current_player)
        target_hand = self._get_hand(target_player)

        if target_index == self.current_player:
            return False
        if self.turn_actions["steal"]:
            return False
        if len(current_hand) >= 20:
            return False
        if len(target_hand) == 0:
            return False
        if card_index < 0 or card_index >= len(target_hand):
            return False

        return True

    #function for stealing a card
    def steal_card(self, target_index, card_index):

        if not self.can_steal_card(target_index, card_index):
            return False
        current_player = self.players[self.current_player]
        target_player = self.players[target_index]

        current_hand = self._get_hand(current_player)
        target_hand = self._get_hand(target_player)

        stolen_card = target_hand.pop(card_index)
        current_hand.append(stolen_card)

        self.turn_actions["steal"] = True
        
        self.check_winner()

        return True

    #check whether the actions lead to a win
    def check_winner(self):
        for i,player in enumerate(self.players):
            hand = self._get_hand(player)

            if len(hand) == 0:
                self.winner = i
                self.game_over = True
                return True 

        return False

    #check whether a discard is possible
    def can_discard_card(self, card_index):
        
        if self.game_over:
            return False
            
        player = self.players[self.current_player]
        hand = self._get_hand(player)
       
        if self.turn_actions["single_discards"] >= 2:
            return False

        if card_index < 0 or card_index >= len(hand):
            return False

        return True

    #function for discarding cards
    def discard_card(self, card_index):
        
        if not self.can_discard_card(card_index):
            return False
        player = self.players[self.current_player]
        hand = self._get_hand(player)
        
        discarded_card = hand.pop(card_index)

        self.deck.append(discarded_card)

        random.shuffle(self.deck)

        self.turn_actions["single_discards"] += 1

        self.check_winner()

        return True        

    #check whether a group of cards is a valid set
    def is_valid_group(self, selected_cards):
        
        if len(selected_cards) < 3:
            return False
        numbers = [card.number for card in selected_cards]

        if len(set(numbers)) == 1:
            return True
            
        sorted_numbers = sorted(numbers)

        for i in range(1, len(sorted_numbers)):
            if sorted_numbers[i] != sorted_numbers[i-1] + 1:
                return False
        colours = {card.colour for card in selected_cards}
        if len(colours)>2:
            return False

        return True                               

    #discard a valid set
    def discard_group(self, selected_cards):
        if self.game_over:
            return False
        player = self.players[self.current_player]
        hand = self._get_hand(player)

        if not self.is_valid_group(selected_cards):
            return False
        remaining_hand = hand.copy()

        for card in selected_cards:
            if card not in remaining_hand:
                return False
            remaining_hand.remove(card)

        for card in selected_cards:
            hand.remove(card)
            
        self.deck.extend(selected_cards)

        random.shuffle(self.deck)
        
        self.check_winner()

        return True
        

    #end of turn
    def end_turn(self):

        if self.game_over:
            return False

        self.current_player = (self.current_player + 1) % len(self.players)

        self.start_turn()

        return True
